# Farm Data Preparation

This notebook prepares the farm dataset used in the project by combining several sheep-related indicators into one structured table.

The raw data come from separate Excel files and include the following variables:

- **lambs**
- **rams**
- **ewes**
- **lamb_sheep_shorn**
- **sheep_flock**
- **sheep_purchased**

The main goal of this notebook is to:

1. load the raw farm files;
2. standardize variable names;
3. extract and harmonize state information;
4. aggregate values by year and state;
5. merge all farm indicators into one dataset;
6. save the final output as `farm.csv`.

In [1]:
import pandas as pd
from functools import reduce

In [2]:
farm_lambs = pd.read_excel("../raw data/lambs.xlsx", header=2)
farm_rams = pd.read_excel("../raw data/rams.xlsx", header=2)
farm_ewes = pd.read_excel("../raw data/ewes.xlsx", header=2)
farm_lamb_sheep_shorn = pd.read_excel("../raw data/sheep and lambs shorn.xlsx", header=2)
farm_sheep_flock = pd.read_excel("../raw data/sheep flock.xlsx", header=2)
farm_sheep_purchased = pd.read_excel("../raw data/sheep purchased.xlsx", header=2)

In [3]:
farm_lambs.head()

,ABARES Region,Сумма val_reg,sort_year
0,NSW Central West,609,2024
1,NSW Central West,580,2023
2,NSW Central West,633,2022
3,NSW Central West,565,2021
4,NSW Central West,398,2020


In [4]:
# Store all farm datasets in a list for consistent cleaning
farm = [farm_lambs, farm_rams, farm_ewes, farm_lamb_sheep_shorn, farm_sheep_flock, farm_sheep_purchased]

# Standardize key column names across all datasets
for df in farm:
    df.rename(columns={"Сумма val_reg": "number", "sort_year" : "year"}, inplace=True)

In [5]:
# Inspect the regional naming format in the raw data
farm_lambs["ABARES Region"].unique()

array(['NSW Central West', 'NSW Coastal', 'NSW Far West',
       'NSW North West Slopes and Plains', 'NSW Riverina',
       'NSW Tablelands (Northern Central and Southern)',
       'NT Alice Springs Districts', 'NT Barkly Tablelands',
       'NT Top End Darwin', 'NT Victoria River District - Katherine',
       'QLD Cape York and the Gulf', 'QLD Central North',
       'QLD Charleville - Longreach', 'QLD Eastern Darling Downs',
       'QLD Northern Coastal - Mackay to Cairns',
       'QLD Southern Coastal - Curtis to Moreton',
       'QLD West and South West',
       'QLD Western Downs and Central Highlands', 'SA Eyre Peninsula',
       'SA Murray Lands and Yorke Peninsula', 'SA Northern Pastoral',
       'SA South East', 'TAS Tasmania', 'VIC Central North', 'VIC Mallee',
       'VIC Southern and Eastern Victoria', 'VIC Wimmera',
       'WA Central and Southern Wheat Belt',
       'WA Northern and Eastern Wheat Belt',
       'WA Pilbara and Central Pastoral', 'WA South West Coastal',
   

In [6]:
# Map state abbreviations to full state names
state_map = {
    "NSW": "New South Wales",
    "NT":  "Northern Territory",
    "QLD": "Queensland",
    "SA":  "South Australia",
    "TAS": "Tasmania",
    "VIC": "Victoria",
    "WA":  "Western Australia",
}

In [7]:
# Extract the state abbreviation from "ABARES Region", convert it to a full state name, and drop the original column
for df in farm:
    df["state"] = df["ABARES Region"].astype(str).str.strip().str.extract(r"^([A-Z]{2,3})\b")[0].map(state_map)
    df.drop(columns=["ABARES Region"], inplace=True)

In [8]:
# Define merge keys
keys = ["year", "state"]

# Assign a final variable name to each dataset
named = {
    "lambs": farm_lambs,
    "rams": farm_rams,
    "ewes": farm_ewes,
    "lamb_sheep_shorn": farm_lamb_sheep_shorn,
    "sheep_flock": farm_sheep_flock,
    "sheep_purchased": farm_sheep_purchased,
}

In [9]:
# Aggregate each dataset by year and state
# Then rename the value column according to the variable name
dfs = []
for col, df in named.items():
    tmp = (df.groupby(keys, as_index=False)["number"].sum().rename(columns={"number": col}))
    
    # Convert values to numeric and keep nullable integer format
    tmp[col] = pd.to_numeric(tmp[col], errors="coerce").round().astype("Int64")
    dfs.append(tmp)

In [10]:
# Merge all farm indicators into one final dataset
farm_all = reduce(lambda l, r: l.merge(r, on=keys, how="outer"), dfs)
farm_all

,year,state,lambs,rams,ewes,lamb_sheep_shorn,sheep_flock,sheep_purchased
0,1990,New South Wales,4073,206,8649,18393,17123,976
1,1990,Northern Territory,0,0,2,15,4,15
2,1990,Queensland,3856,157,8095,17240,17443,1023
3,1990,South Australia,2803,145,5276,11211,10952,941
4,1990,Tasmania,641,34,1141,2813,2589,86
...,...,...,...,...,...,...,...,...
254,2026,Queensland,<NA>,<NA>,<NA>,<NA>,5908,<NA>
255,2026,South Australia,<NA>,<NA>,<NA>,<NA>,10755,<NA>
256,2026,Tasmania,<NA>,<NA>,<NA>,<NA>,1999,<NA>
257,2026,Victoria,<NA>,<NA>,<NA>,<NA>,4453,<NA>


In [11]:
farm_all.to_csv("../datasets/farm.csv", index=False)